# Would you approve it?

Last week you built a model. It looks at a person's gut bacteria and blood results
and predicts whether they will be diagnosed with type 2 diabetes within one year. The
AUC was about 0.79. That is a respectable number, and I have been looking at it for a
week, trying to understand what it actually *means*.

Here is the question I cannot answer with that number: **could this be a test?**

Not a paper. A test. Something a doctor orders, a laboratory runs, and a result comes
back. That is very different from "a model with a good AUC", and the difference
between the two is what this afternoon is about.

One thing to know before we start: the model does not change. Not once. Everything
that changes from here on is something the model did not control.

## How this works

One model, fitted once in section 1. It produces 700 scores, and those scores stay
exactly the same for the rest of the notebook. After that, each section changes
something *around* the model - where the line is drawn, who is in the room, which
average we use - and we watch how the metrics change.

Where you see **Task**, the cell below it is for you to write. There is a hint in the
comment: usually the function you need, or the pattern from an earlier week. Cells
marked *supplied* you only need to run. If you get stuck on a task, skip it and keep
going. The notebook still runs.

**In Google Colab:** use a Python CPU runtime. Run the setup cell before the other cells. The CSV files load directly from GitHub. Use **File → Save a copy in Drive** to keep your work, then download the completed `.ipynb` for submission.

[Session 4 handout and intended-use cards](https://github.com/nbrg-ppcu/appliedmedtech/blob/session-04-metrics/notebooks/session_04/S4_lab_handout.md).

| Dataset | CSV | Data dictionary |
|---|---|---|
| Development cohort | [Download CSV](https://raw.githubusercontent.com/nbrg-ppcu/appliedmedtech/main/notebooks/session_04/data/t2d_microbiome_clean.csv) | [Read dictionary](https://github.com/nbrg-ppcu/appliedmedtech/blob/session-04-metrics/notebooks/session_04/data/t2d_microbiome_clean_DICTIONARY.md) |
| Screening cohort | [Download CSV](https://raw.githubusercontent.com/nbrg-ppcu/appliedmedtech/main/notebooks/session_04/data/t2d_screening.csv) | [Read dictionary](https://github.com/nbrg-ppcu/appliedmedtech/blob/session-04-metrics/notebooks/session_04/data/t2d_screening_DICTIONARY.md) |
| Emergency admissions | [Download CSV](https://raw.githubusercontent.com/nbrg-ppcu/appliedmedtech/main/notebooks/session_04/data/ed_deterioration.csv) | [Read dictionary](https://github.com/nbrg-ppcu/appliedmedtech/blob/session-04-metrics/notebooks/session_04/data/ed_deterioration_DICTIONARY.md) |


## 1 · Setup and the model  *(supplied)*

Nothing to write here. Run the cells and check that the numbers make sense.

In [ ]:
%pip install -q numpy pandas scikit-learn matplotlib


In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import (confusion_matrix, accuracy_score, recall_score, precision_score,
                             f1_score, balanced_accuracy_score, matthews_corrcoef,
                             roc_auc_score, roc_curve, average_precision_score)

SEED = 42
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
display("Ready.")

Last week's cohort, but already cleaned, so you do not need to repeat last week's
work. `bmi` is numeric, `hba1c_pct` is in percent at every site, sex has two values
and smoking has three. The columns are the same as in the screening file you will
meet in section 5. That is important: one model, two datasets, no changes in between.

In [ ]:
dev = pd.read_csv("https://raw.githubusercontent.com/nbrg-ppcu/appliedmedtech/main/notebooks/session_04/data/t2d_microbiome_clean.csv")
display(pd.Series({
    "rows": dev.shape[0],
    "columns": dev.shape[1],
    "prevalence": round(dev["progressed_12m"].mean(), 3),
}, name="development cohort"))
display(dev.head(3))

Your column decisions from last week, and your pipeline, exactly as you left them.
The two study-record fields are still dropped - we decided that last week.

In [ ]:
TARGET = "progressed_12m"
DROP_COLS = ["participant_id", "protocol_version", "homa_ir", "extraction_kit",
             "metformin_initiated", "clinic_referral"]          # the two record fields: filled at diagnosis
features = [c for c in dev.columns if c not in DROP_COLS + [TARGET]]
NUMERIC  = [c for c in features if pd.api.types.is_numeric_dtype(dev[c])]
ORDERED  = ["physical_activity"]
NOMINAL  = [c for c in features if c not in NUMERIC + ORDERED]

numeric_steps = Pipeline([("impute", SimpleImputer(strategy="median", add_indicator=True)),
                          ("scale",  StandardScaler())])
nominal_steps = Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                          ("encode", OneHotEncoder(handle_unknown="ignore"))])
ordered_steps = Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                          ("encode", OrdinalEncoder(categories=[["low","moderate","high"]],
                                     handle_unknown="use_encoded_value", unknown_value=-1))])
preprocessor = ColumnTransformer([("num", numeric_steps, NUMERIC),
                                  ("nom", nominal_steps, NOMINAL),
                                  ("ord", ordered_steps, ORDERED)])
def build(model):
    return Pipeline([("prep", preprocessor), ("model", model)])

X, y = dev[features], dev[TARGET].values
display(f"{len(features)} features")

**These 700 scores are the whole notebook.** Every person is scored by the single
model fitted on the full development cohort. The scores will not change. Everything
that follows is a different way of *reading* them.

In [ ]:
model = build(LogisticRegression(max_iter=4000)).fit(X, y)
scores = model.predict_proba(X)[:, 1]

display(pd.Series({
    "scores for the first six people": scores[:6].round(3).tolist(),
    "AUC": round(roc_auc_score(y, scores), 3),
}))

## 2 · Seven numbers by hand

Before sklearn does anything for us, we will do it once ourselves. Every metric in the
lecture is a ratio of four integers, and you should compute each one yourself at least
once.

*Supplied.* A score is not a decision. Somebody has to draw a line. I have drawn it
where the model catches 80% of the people who progress. We will question that choice
in section 3. For now, this is the line.

In [ ]:
THRESHOLD = 0.208
flagged = (scores >= THRESHOLD).astype(int)
display(f"flagged: {flagged.sum()} of {len(flagged)}")

### Task 1 · Two piles
Cross what actually happened against what the model said. You will get a 2×2 table.
Name the four cells: which one is TP, FP, FN, TN? Rows are the truth, columns are the
model.

In [ ]:
# TASK 1
# Hint:  pd.crosstab(y, flagged)   - rows are the truth, columns are the model



### Task 2 · Four ratios, by hand
From those four cells, compute accuracy, sensitivity, specificity and precision as
plain arithmetic - no sklearn yet. *Then* check each one against sklearn. If they
disagree, one of the fractions is the wrong way round. It is usually not sklearn.

In [ ]:
# TASK 2
# Hint:  accuracy = (TP + TN) / (TP + TN + FP + FN)
#        sensitivity = TP / (TP + FN)      specificity = TN / (TN + FP)      precision = TP / (TP + FP)
# Check: accuracy_score(y, flagged), recall_score(y, flagged), recall_score(1-y, 1-flagged), precision_score(y, flagged)



### Task 3 · Three more
Now F1, balanced accuracy and MCC, with sklearn. Then - this is the important part -
compute F1 **again** with `pos_label=0`, so that "did not progress" is treated as the
positive class. Same predictions, same threshold. Watch what happens.

In [ ]:
# TASK 3
# Hint:  f1_score(y, flagged)   balanced_accuracy_score(y, flagged)   matthews_corrcoef(y, flagged)
#        f1_score(y, flagged, pos_label=0)



### Task 4 · The do-nothing model
Make a prediction that says "no" for every single person - a model that has learned
nothing. Compute all seven metrics for it and compare each one with
your model. Does any of them rank the do-nothing model above yours?

In [ ]:
# TASK 4
# Hint:  nothing = np.zeros_like(y)   then the same seven calls with `nothing` in place of `flagged`



### Task 5 · The coin flip
The do-nothing model is one kind of baseline. A coin is the other. Make a prediction
that flips a fair coin for each person, and a *score* that is just random noise.
Compute all seven metrics plus AUC. Which metrics correctly say "this is chance"? And
look carefully at the precision: what number is it, and why that number?

In [ ]:
# TASK 5
# Hint:  rng = np.random.default_rng(SEED)
#        coin = rng.integers(0, 2, len(y));  noise = rng.random(len(y))
#        the seven calls with `coin`, plus roc_auc_score(y, noise)



**Your conclusion.** In one or two sentences: what does this section show? Write it in this cell.

## 3 · You are responsible for the threshold

I drew the line at 0.208 in section 2 without asking you. That was a decision, and the
model had no part in it. From here on it is your decision, and you are responsible for
it. Let us see what the options are. 

*Supplied.* Two hundred possible lines, and what four metrics do as the line moves
from left to right. This plot is the whole section.

In [ ]:
ths = np.linspace(0.02, 0.98, 200)
sweep = pd.DataFrame({"threshold": ths})
sweep["sensitivity"] = [recall_score(y, scores >= t) for t in ths]
sweep["specificity"] = [recall_score(1-y, scores < t) for t in ths]
sweep["precision"]   = [precision_score(y, scores >= t, zero_division=0) for t in ths]
sweep["F1"]          = [f1_score(y, scores >= t, zero_division=0) for t in ths]

plt.figure(figsize=(9, 4.5))
for col in ["sensitivity", "specificity", "precision", "F1"]:
    plt.plot(sweep["threshold"], sweep[col], label=col, lw=2)
plt.xlabel("threshold"); plt.ylabel("metric"); plt.legend(frameon=False); plt.grid(alpha=.25)
plt.title("Same model, every threshold"); plt.show()

### Task 6 · Read the sweep
At what threshold does precision first reach 0.60? And what has happened to
sensitivity at that point?

In [ ]:
# TASK 5
# Hint:  sweep[sweep["precision"] >= 0.60].iloc[0]



### Task 7 · Two deployments, one model
**Scenario A - a screening programme.** Missing somebody costs them a year of
prevention. Choose the threshold that catches 80% of progressors.
**Scenario B - confirming before starting a drug with side effects.** A false alarm
means a prescription that somebody did not need. Choose the threshold where precision
reaches 0.60.
Print the confusion matrix for each. Same model. See how different the two tests are.

In [ ]:
# TASK 6
# Hint:  iA = np.argmin(np.abs(sweep["sensitivity"] - 0.80));  tA = sweep["threshold"][iA]
#        then confusion_matrix(y, scores >= tA)



### Task 8 · ROC
Plot the ROC curve with `roc_curve`, mark your two thresholds on it, and display the AUC.
You have reported AUC for three weeks. This is the first time you will see what is
behind it.

In [ ]:
# TASK 7
# Hint:  fpr, tpr, thr = roc_curve(y, scores);  plt.plot(fpr, tpr)
#        to mark a threshold t: i = np.argmin(np.abs(thr - t)); plt.scatter(fpr[i], tpr[i])



**Your conclusion.** In one or two sentences: what does this section show? Write it in this cell.

## 4 · Same predictions, three rooms

Here is a trick that seems like cheating, but is not. We keep every prediction exactly
as it is, and only change who is in the room - by resampling the negatives until the
prevalence is whatever we choose. The model does not know.

*Supplied.* Keep every positive, resample the negatives to reach a target prevalence,
and return the same predictions for the new room.

In [ ]:
rng = np.random.default_rng(SEED)
def in_a_room(prevalence):
    pos = np.where(y == 1)[0]; neg = np.where(y == 0)[0]
    n_neg = int(len(pos) * (1 - prevalence) / prevalence)
    idx = np.concatenate([pos, rng.choice(neg, n_neg, replace=True)])
    return y[idx], flagged[idx]

### Task 9 · Three prevalences
For 28%, 10% and 1%: display sensitivity, specificity, precision, accuracy - and the
do-nothing accuracy next to them. Which numbers change with the prevalence, and which
do not?

In [ ]:
# TASK 8
# Hint:  for prev in [0.28, 0.10, 0.01]:
#            yr, fr = in_a_room(prev)
#            ... recall_score(yr, fr), recall_score(1-yr, 1-fr), precision_score(yr, fr), accuracy_score(yr, fr), 1 - prev



**Your conclusion.** In one or two sentences: what does this section show? Write it in this cell.

## 5 · The screening programme

Section 4 was a simulation. This is the real situation - or as close to real as
synthetic data can be. Two thousand prediabetic adults found at routine GP screening,
followed for one year. This is the population the test was *made for*. Let us find
out what it does there.

*Supplied.* Same columns, already harmonised by the screening programme's data
office, so your fitted model runs on it directly. Check the prevalence before you
continue.

### Task 10 · Take the model to the clinic
Use `model.predict_proba` on the screening cohort. Keep the **same 0.208 threshold** -
we chose it on the development cohort, and now we will find out whether that was a
good idea. Compute all seven metrics plus AUC and PR-AUC, for both cohorts, side by
side in one table.

In [ ]:
# TASK 9
# Hint:  scores_s = model.predict_proba(X_s)[:, 1];  flagged_s = (scores_s >= THRESHOLD).astype(int)
#        build a dict of metric -> (dev value, screening value) and pd.DataFrame it
#        PR-AUC is average_precision_score(y, scores)



### Task 11 · Which numbers survived the move?
Sort the rows of your table into three groups: **held**, **rose**, **fell**. Is there a
row that surprises you, given what the lecture said? Note it.

In [ ]:
scr = pd.read_csv("https://raw.githubusercontent.com/nbrg-ppcu/appliedmedtech/main/notebooks/session_04/data/t2d_screening.csv")
X_s, y_s = scr[features], scr[TARGET].values
display(pd.Series({
    "rows": len(scr),
    "prevalence": round(y_s.mean(), 3),
}, name="screening cohort"))

In [ ]:
# TASK 11 - write your three groups as a comment, or display them
held = []
rose = []
fell = []

### Task 12 · A closer look at the true positives
Compare the mean score of the *true positives* in the two cohorts. What does the
difference tell you about the people in each cohort? Then choose the threshold again,
on the screening cohort, so that it catches 80%. Display how many people the programme
now has to flag, and how many of those are real.

In [ ]:
# TASK 11
# Hint:  scores[y == 1].mean()   vs   scores_s[y_s == 1].mean()
#        fpr_s, tpr_s, thr_s = roc_curve(y_s, scores_s);  i = np.argmin(np.abs(tpr_s - 0.80))
#        flagged_new = scores_s >= thr_s[i];  flagged_new.sum(), (flagged_new & (y_s == 1)).sum()



**Your conclusion.** In one or two sentences: what does this section show? Write it in this cell.

## 6 · Four classes, one rare

Back to the emergency department table from two weeks ago, with a new target: where
did the patient end up? Home, the ward, intensive care, or dead. Four classes. You can
guess which one is rare, and which one matters most.

*Supplied.* Cleaning as before. Look at the class counts before you continue.

In [ ]:
ed = pd.read_csv("https://raw.githubusercontent.com/nbrg-ppcu/appliedmedtech/main/notebooks/session_04/data/ed_deterioration.csv")
ed["temp_c"] = pd.to_numeric(ed["temp_c"].astype(str).str.replace(",", ".", regex=False), errors="coerce")
ed["sex_c"] = ed["sex"].str.upper().str[0]
ed["creatinine"] = ed["creatinine"].where(ed["site"] != "C", ed["creatinine"] / 88.4)
ED_NUM = ["age","bmi","heart_rate","resp_rate","sbp","temp_c","spo2","wbc","creatinine","crp","lactate","comorbidity_count"]
ED_CAT = ["site","sex_c","charlson_band"]
ed_pre = ColumnTransformer([("num", Pipeline([("i", SimpleImputer(strategy="median", add_indicator=True)), ("s", StandardScaler())]), ED_NUM),
                            ("cat", Pipeline([("i", SimpleImputer(strategy="most_frequent")), ("o", OneHotEncoder(handle_unknown="ignore"))]), ED_CAT)])
X_ed, y_ed = ed[ED_NUM + ED_CAT], ed["discharge_disposition"].values
LABELS = ["home", "ward", "ICU", "died"]
cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
display(pd.Series(y_ed).value_counts().reindex(LABELS))

### Task 13 · Per-class recall
Cross-validate predictions with a logistic pipeline, then display recall **for each class
separately**. Add accuracy, and the do-nothing accuracy (always predict the most common
class). Look at the rare class first.

In [ ]:
# TASK 12
# Hint:  pred_ed = cross_val_predict(Pipeline([("prep", ed_pre), ("model", LogisticRegression(max_iter=4000))]), X_ed, y_ed, cv=cv)
#        recall_score(y_ed, pred_ed, labels=LABELS, average=None)



### Task 14 · Three ways to average
`f1_score` with `average="macro"`, then `"micro"`, then `"weighted"`. Same predictions,
three numbers. Compare each one with the accuracy from Task 13. What does each one tell you
about the rare class?

In [ ]:
# TASK 13
# Hint:  f1_score(y_ed, pred_ed, average="macro")   and "micro", "weighted"



### Task 15 · Weighting the classes
Refit with `class_weight="balanced"` and run Tasks 13 and 14 again. What changes for the
rare class? What changes for the other classes? Would you call this an improvement - and
does your answer depend on which average you report?

In [ ]:
# TASK 14
# Hint:  LogisticRegression(max_iter=4000, class_weight="balanced")   then the same two calls as Tasks 13 and 14



**Your conclusion.** In one or two sentences: what does this section show? Write it in this cell.

## 7 · The memo

Your intended-use card is on the handout. Use the **screening cohort** predictions
from section 5, fill in the form below, give a verdict, and **upload this notebook to
Moodle**. Half a page of reasoning is enough. The most honest answer wins, even if it
is the lowest number. If your card says reject and the numbers support you, reject.

Two hints:
- Your card's prevalence may not be the screening cohort's 6.9%. You already have
  sensitivity and specificity. Think back to the 2×2 table from the lecture.
- For an interval: resample the rows with replacement many times, recompute your number
  each time, and keep the middle 95%.

| | |
|---|---|
| Intended use, and its prevalence | |
| The metric that matters here, and why | |
| The cut-off you chose, and why | |
| At that cut-off, in this population: sensitivity, specificity, precision | |
| How many people flagged per true case | |
| The do-nothing comparison, on the same metric | |
| An interval on your main number | |
| **Verdict** | approve / approve with condition / reject |
| The condition, or the reason | |

In [ ]:
# scratch - a place to work on your memo
